# Examen blanc B : GRU ou encodeur Transformer pour reconnaitre une activite ?

**Tache** : donnees du TP4 (Human Activity Recognition), series temporelles 128 pas x 9 capteurs,
6 classes d'activite.

**Questions** :
1. Un encodeur Transformer (self-attention) fait-il mieux, moins bien ou pareil qu'un GRU,
   a budget comparable ?
2. L'encodage de position est-il necessaire pour le Transformer ? (sans lui, l'attention
   voit la sequence comme un **ensemble** : l'ordre des pas de temps est perdu)

**Modeles** (memes splits, meme optimiseur, memes epochs, 3 seeds) :
- `gru` : GRU 1 couche, 150 unites (~73k parametres, comme le Transformer), on classe a partir de h_n (comme le LSTM du TP4) ;
- `transformer` : projection lineaire 9 -> 64, + encodage de position appris, + token [CLS],
  2 couches d'encodeur, classification sur le vecteur de [CLS] (comme BERT) ;
- `transformer_nopos` : identique, sans encodage de position (ablation).

In [ ]:
import os, time, random, copy
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")
QUICK = False  # True pour un test tres rapide (CPU)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

## 1. Donnees (fonction `load_dataset` du TP4)

In [ ]:
def load_dataset(train, path='data'):
    import zipfile
    zip1 = os.path.join(path, 'human+activity+recognition+using+smartphones.zip')
    if not os.path.isfile(zip1):
        from urllib.request import urlretrieve
        os.makedirs(path, exist_ok=True)
        urlretrieve('https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip', zip1)
    if not os.path.isfile(os.path.join(path, 'UCI HAR Dataset.zip')):
        with zipfile.ZipFile(zip1, 'r') as z:
            z.extractall(path)
    if not os.path.isdir(os.path.join(path, 'UCI HAR Dataset')):
        with zipfile.ZipFile(os.path.join(path, 'UCI HAR Dataset.zip'), 'r') as z:
            z.extractall(path)
    set_ = 'train' if train else 'test'
    names = [f'{s}_{a}_{set_}.txt' for s in ('body_acc', 'body_gyro', 'total_acc') for a in 'xyz']
    X = np.asarray([np.loadtxt(os.path.join(path, 'UCI HAR Dataset', set_, 'Inertial Signals', f)) for f in names])
    X = np.transpose(X, (1, 2, 0))                                   # (n, 128, 9)
    y = np.loadtxt(os.path.join(path, 'UCI HAR Dataset', set_, f'y_{set_}.txt'))
    subject = np.loadtxt(os.path.join(path, 'UCI HAR Dataset', set_, f'subject_{set_}.txt'), dtype='int64')
    return X, y, subject


X_train, y_train, subject_train = load_dataset(train=True)
X_test, y_test, subject_test = load_dataset(train=False)
print(X_train.shape, X_test.shape)

**Split de validation par sujet.** Le test contient des personnes jamais vues. Pour que la
validation mesure la meme chose (generaliser a de nouvelles personnes), on met de cote
5 sujets entiers du train, plutot qu'un `random_split` ligne a ligne : des fenetres d'une
meme personne qui se chevauchent a 50 % seraient sinon a la fois en train et en validation.

In [ ]:
subjects = np.unique(subject_train)
val_subjects = np.random.RandomState(0).choice(subjects, 5, replace=False)
is_val = np.isin(subject_train, val_subjects)
print("Sujets de validation :", sorted(val_subjects), "| fenetres :", is_val.sum())

# Standardisation par canal, avec les statistiques du train seulement (pas de fuite)
mean = X_train[~is_val].mean(axis=(0, 1), keepdims=True)
std = X_train[~is_val].std(axis=(0, 1), keepdims=True)


def to_tensors(X, y):
    X = torch.from_numpy((X - mean) / std).to(dtype=torch.float32)
    y = torch.from_numpy(y - 1).to(dtype=torch.int64)               # classes 1..6 -> 0..5
    return X, y


class CustomDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_data = CustomDataset(*to_tensors(X_train[~is_val], y_train[~is_val]))
val_data = CustomDataset(*to_tensors(X_train[is_val], y_train[is_val]))
test_data = CustomDataset(*to_tensors(X_test, y_test))
val_dataloader = DataLoader(val_data, batch_size=256)
test_dataloader = DataLoader(test_data, batch_size=256)

## 2. Modeles

In [ ]:
class Classifier(nn.Module):
    """Partie commune : optimiseur et pas d'entrainement/validation (design du TP4)."""

    def __init__(self):
        super().__init__()
        self.loss_fn_mean = nn.CrossEntropyLoss(reduction='mean')
        self.loss_fn_sum = nn.CrossEntropyLoss(reduction='sum')

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-3)

    def training_step(self, batch):
        X, y = batch
        return self.loss_fn_mean(self(X), y)

    def validation_step(self, batch):
        X, y = batch
        with torch.no_grad():
            pred = self(X)
            loss = self.loss_fn_sum(pred, y).item()
        return loss, pred.argmax(1), y


class GRUNetwork(Classifier):
    def __init__(self, n_channels=9, n_classes=6, n_hidden=150):
        super().__init__()
        self.gru = nn.GRU(n_channels, n_hidden, batch_first=True)
        self.linear = nn.Linear(n_hidden, n_classes)

    def forward(self, x):                       # x : (batch, 128, 9)
        _, h_n = self.gru(x)                    # h_n : (1, batch, 150), etat apres le dernier pas
        return self.linear(h_n[-1])


class TransformerNetwork(Classifier):
    def __init__(self, n_channels=9, n_classes=6, d_model=64, n_heads=4, n_layers=2,
                 seq_len=128, use_position=True):
        super().__init__()
        self.input_proj = nn.Linear(n_channels, d_model)             # "embedding" de chaque pas
        self.cls = nn.Parameter(torch.zeros(1, 1, d_model))          # token [CLS] appris
        self.use_position = use_position
        self.pos = nn.Parameter(torch.randn(1, seq_len + 1, d_model) * 0.02)  # position apprise
        layer = nn.TransformerEncoderLayer(d_model, n_heads, dim_feedforward=128,
                                           dropout=0.1, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, n_layers)
        self.linear = nn.Linear(d_model, n_classes)

    def forward(self, x):                       # x : (batch, 128, 9)
        h = self.input_proj(x)                  # (batch, 128, 64)
        cls = self.cls.expand(h.size(0), -1, -1)
        h = torch.cat([cls, h], dim=1)          # (batch, 129, 64), [CLS] en position 0
        if self.use_position:
            h = h + self.pos                    # embedding = contenu + position (comme BERT)
        h = self.encoder(h)                     # self-attention sur les 129 positions
        return self.linear(h[:, 0])             # vecteur de [CLS]


def make_model(name):
    return {"gru": lambda: GRUNetwork(),
            "transformer": lambda: TransformerNetwork(use_position=True),
            "transformer_nopos": lambda: TransformerNetwork(use_position=False)}[name]()

## 3. Entrainement (Trainer du TP4, avec historique et meilleure epoch en validation)

In [ ]:
def evaluate(dataloader, model):
    model.eval()
    size = len(dataloader.dataset)
    loss, correct = 0., 0.
    for X, y in dataloader:
        X, y = X.to(device), y.to(device)
        batch_loss, pred, y_true = model.validation_step((X, y))
        loss += batch_loss
        correct += (pred == y_true).type(torch.float).sum().item()
    return loss / size, correct / size


class Trainer():
    def __init__(self, max_epochs):
        self.max_epochs = max_epochs

    def fit(self, model, train_dataloader, val_dataloader):
        model = model.to(device)
        optimizer = model.configure_optimizers()
        history = {"train_loss": [], "val_loss": [], "val_acc": []}
        best_acc, best_state = -1., None
        for epoch in range(self.max_epochs):
            model.train()
            running, n_batches = 0., 0
            for X, y in train_dataloader:
                X, y = X.to(device), y.to(device)
                loss = model.training_step((X, y))
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                running += loss.item()
                n_batches += 1
            val_loss, val_acc = evaluate(val_dataloader, model)
            history["train_loss"].append(running / n_batches)
            history["val_loss"].append(val_loss)
            history["val_acc"].append(val_acc)
            if val_acc > best_acc:
                best_acc, best_state = val_acc, copy.deepcopy(model.state_dict())
        model.load_state_dict(best_state)
        return history


def run(name, seed, max_epochs):
    set_seed(seed)
    g = torch.Generator().manual_seed(seed)
    data = train_data
    if QUICK:
        data = torch.utils.data.Subset(train_data, range(600))
    train_dataloader = DataLoader(data, batch_size=64, shuffle=True, generator=g)
    model = make_model(name)
    n_params = sum(p.numel() for p in model.parameters())
    t0 = time.time()
    history = Trainer(max_epochs).fit(model, train_dataloader, val_dataloader)
    return model, history, n_params, time.time() - t0

## 4. Grille : 3 modeles x 3 seeds, memes epochs

In [ ]:
MODELS = ["gru", "transformer", "transformer_nopos"]
SEEDS = [0] if QUICK else [0, 1, 2]
EPOCHS = 1 if QUICK else 15

results, histories = [], {}
for name in MODELS:
    for seed in SEEDS:
        model, history, n_params, secs = run(name, seed, EPOCHS)
        histories[(name, seed)] = history
        results.append((name, seed, max(history["val_acc"]), n_params, secs))
        print(f"{name:18s} seed={seed} best val_acc={max(history['val_acc']):.3f} "
              f"params={n_params:,} ({secs:.0f} s)")

In [ ]:
import pandas as pd

df = pd.DataFrame(results, columns=["model", "seed", "val_acc", "n_params", "seconds"])
print(df.groupby("model").agg(mean=("val_acc", "mean"), std=("val_acc", "std"),
                              n_params=("n_params", "first"), seconds=("seconds", "mean")).round(3))

fig, axes = plt.subplots(1, len(MODELS), figsize=(14, 3.5), sharey=True)
for ax, name in zip(axes, MODELS):
    for seed in SEEDS:
        h = histories[(name, seed)]
        ax.plot(h["train_loss"], color="C0", alpha=.7, label="train" if seed == 0 else None)
        ax.plot(h["val_loss"], color="C1", alpha=.7, label="val" if seed == 0 else None)
    ax.set_title(name); ax.set_xlabel("epoch")
axes[0].set_ylabel("loss"); axes[0].legend(); plt.show()

## 5. Test : une seule fois, a la fin

In [ ]:
for name in MODELS:
    model, _, _, _ = run(name, seed=0, max_epochs=EPOCHS)
    _, test_acc = evaluate(test_dataloader, model)
    print(f"TEST {name:18s}: accuracy {100 * test_acc:.1f}%")